# **Project Lab: Spot Instance Bidding Strategy (Student)**
##### Copyright by UIT-NC@NT549


### Setup and Imports


In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from collections import deque
import random


## Background: Environment Components


In [ ]:
# -*- coding: utf-8 -*-
"""
Data Generator cho Spot Instance Bidding Strategy

Module này chứa các class để sinh dữ liệu giá spot và workload
với các pattern thực tế như:
- Giá spot thay đổi theo giờ trong ngày
- Giá spot thay đổi theo ngày trong tuần
- Workload với các pattern: stable, spike, random, periodic
"""

import numpy as np
import pandas as pd
from typing import Optional, List, Tuple
from dataclasses import dataclass


@dataclass
class PriceConfig:
    """Cấu hình cho việc sinh giá spot"""
    base_price: float = 0.03        # Giá cơ bản ($/hour)
    max_price: float = 0.08         # Giá tối đa
    min_price: float = 0.01         # Giá tối thiểu
    daily_amplitude: float = 0.02   # Biên độ dao động theo ngày
    weekly_amplitude: float = 0.01  # Biên độ dao động theo tuần
    noise_std: float = 0.005        # Độ lệch chuẩn nhiễu


@dataclass
class WorkloadConfig:
    """Cấu hình cho việc sinh workload"""
    base_workload: int = 10         # Số jobs cơ bản mỗi step
    spike_multiplier: float = 5.0   # Hệ số nhân khi có spike
    spike_probability: float = 0.1  # Xác suất xảy ra spike
    max_workload: int = 100         # Workload tối đa


In [ ]:
class SpotPriceGenerator:
    """
    Class sinh dữ liệu giá spot instance với các pattern thực tế.

    Giá spot thay đổi theo:
    - Giờ trong ngày: Cao hơn vào giờ làm việc (9-17h)
    - Ngày trong tuần: Thấp hơn vào cuối tuần
    - Nhiễu ngẫu nhiên
    """

    def __init__(self, config: Optional[PriceConfig] = None, seed: int = 42):
        """
        Khởi tạo SpotPriceGenerator.

        Args:
            config: Cấu hình giá spot, nếu None sử dụng giá trị mặc định
            seed: Random seed để reproducibility
        """
        self.config = config or PriceConfig()
        self.rng = np.random.default_rng(seed)
        self.current_step = 0

    def reset(self) -> None:
        """Reset generator về trạng thái ban đầu"""
        self.current_step = 0

    def _get_hour_factor(self, hour: int) -> float:
        """
        Tính hệ số giá theo giờ trong ngày.

        Giá cao hơn vào giờ làm việc (9-17h), đỉnh điểm lúc 13h.
        Giá thấp hơn vào ban đêm (0-6h).

        Args:
            hour: Giờ trong ngày (0-23)

        Returns:
            Hệ số nhân giá (0.5 đến 1.5)
        """
        # Sử dụng hàm sin để mô phỏng pattern theo giờ
        # Đỉnh lúc 13h (peak business hours)
        peak_hour = 13
        factor = np.sin(np.pi * (hour - peak_hour + 12) / 24)
        # Scale về khoảng 0.7 đến 1.3
        return 1.0 + 0.3 * factor

    def _get_day_factor(self, day: int) -> float:
        """
        Tính hệ số giá theo ngày trong tuần.

        Giá thấp hơn vào cuối tuần (thứ 7, CN).
        Giá cao nhất vào giữa tuần (thứ 3, 4).

        Args:
            day: Ngày trong tuần (0=Monday, 6=Sunday)

        Returns:
            Hệ số nhân giá (0.8 đến 1.2)
        """
        # Thứ 3 và thứ 4 có giá cao nhất
        # Cuối tuần có giá thấp nhất
        if day in [5, 6]:  # Thứ 7, CN
            return 0.8
        elif day in [2, 3]:  # Thứ 3, 4
            return 1.2
        else:
            return 1.0

    def get_price(self, hour: int, day: int) -> float:
        """
        Lấy giá spot tại thời điểm cụ thể.

        Args:
            hour: Giờ trong ngày (0-23)
            day: Ngày trong tuần (0-6)

        Returns:
            Giá spot ($/hour)
        """
        # Giá cơ bản
        price = self.config.base_price

        # Áp dụng hệ số theo giờ
        hour_factor = self._get_hour_factor(hour)
        price += self.config.daily_amplitude * (hour_factor - 1)

        # Áp dụng hệ số theo ngày
        day_factor = self._get_day_factor(day)
        price += self.config.weekly_amplitude * (day_factor - 1)

        # Thêm nhiễu ngẫu nhiên
        noise = self.rng.normal(0, self.config.noise_std)
        price += noise

        # Giới hạn giá trong khoảng cho phép
        price = np.clip(price, self.config.min_price, self.config.max_price)

        return price

    def generate_price_series(self, n_steps: int, start_hour: int = 0,
                              start_day: int = 0) -> np.ndarray:
        """
        Sinh chuỗi giá spot cho n_steps.

        Args:
            n_steps: Số steps cần sinh
            start_hour: Giờ bắt đầu
            start_day: Ngày bắt đầu

        Returns:
            Array giá spot cho mỗi step
        """
        prices = np.zeros(n_steps)

        for i in range(n_steps):
            # Tính giờ và ngày hiện tại
            total_hours = start_hour + i
            current_hour = total_hours % 24
            current_day = (start_day + total_hours // 24) % 7

            prices[i] = self.get_price(current_hour, current_day)

        return prices

    def step(self) -> Tuple[float, int, int]:
        """
        Tiến một step và trả về giá, giờ, ngày.

        Returns:
            Tuple (price, hour, day)
        """
        hour = self.current_step % 24
        day = (self.current_step // 24) % 7
        price = self.get_price(hour, day)
        self.current_step += 1
        return price, hour, day

    def get_interruption_probability(self, price: float) -> float:
        """
        Tính xác suất bị interrupt dựa trên giá.

        Giá càng cao thì xác suất interrupt càng cao.

        Args:
            price: Giá spot hiện tại

        Returns:
            Xác suất interrupt (0 đến 1)
        """
        # Normalize giá về khoảng 0-1
        price_normalized = (price - self.config.min_price) / \
                          (self.config.max_price - self.config.min_price)

        # Xác suất interrupt tăng theo giá
        # Base prob: 0.02 khi giá thấp nhất
        # Max prob: 0.20 khi giá cao nhất
        base_prob = 0.02
        max_prob = 0.20

        prob = base_prob + (max_prob - base_prob) * price_normalized
        return prob


In [ ]:
class WorkloadGenerator:
    """
    Class sinh workload với các pattern khác nhau.

    Hỗ trợ các pattern:
    - stable: Workload ổn định
    - spike: Có đột biến ngẫu nhiên
    - random: Hoàn toàn ngẫu nhiên
    - periodic: Theo chu kỳ (cao vào ban ngày)
    """

    def __init__(self, config: Optional[WorkloadConfig] = None,
                 pattern: str = "stable", seed: int = 42):
        """
        Khởi tạo WorkloadGenerator.

        Args:
            config: Cấu hình workload
            pattern: Loại pattern ("stable", "spike", "random", "periodic")
            seed: Random seed
        """
        self.config = config or WorkloadConfig()
        self.pattern = pattern
        self.rng = np.random.default_rng(seed)
        self.current_step = 0

        # Validate pattern
        valid_patterns = ["stable", "spike", "random", "periodic"]
        if pattern not in valid_patterns:
            raise ValueError(f"Pattern phải là một trong {valid_patterns}")

    def reset(self) -> None:
        """Reset generator về trạng thái ban đầu"""
        self.current_step = 0

    def _get_stable_workload(self) -> int:
        """
        Sinh workload ổn định với nhiễu nhỏ.

        Returns:
            Số jobs cần xử lý
        """
        noise = self.rng.integers(-2, 3)
        workload = self.config.base_workload + noise
        return max(0, workload)

    def _get_spike_workload(self) -> int:
        """
        Sinh workload với khả năng có spike.

        Returns:
            Số jobs cần xử lý
        """
        base = self._get_stable_workload()

        # Có xác suất xảy ra spike
        if self.rng.random() < self.config.spike_probability:
            spike = int(base * self.config.spike_multiplier)
            return min(spike, self.config.max_workload)

        return base

    def _get_random_workload(self) -> int:
        """
        Sinh workload hoàn toàn ngẫu nhiên.

        Returns:
            Số jobs cần xử lý
        """
        return self.rng.integers(0, self.config.max_workload // 2)

    def _get_periodic_workload(self, hour: int) -> int:
        """
        Sinh workload theo chu kỳ ngày.

        Workload cao vào giờ làm việc, thấp vào ban đêm.

        Args:
            hour: Giờ trong ngày (0-23)

        Returns:
            Số jobs cần xử lý
        """
        # Đỉnh workload lúc 10-14h
        if 10 <= hour <= 14:
            multiplier = 2.0
        elif 8 <= hour <= 18:
            multiplier = 1.5
        elif 6 <= hour <= 22:
            multiplier = 1.0
        else:
            multiplier = 0.3

        workload = int(self.config.base_workload * multiplier)
        noise = self.rng.integers(-2, 3)

        return max(0, min(workload + noise, self.config.max_workload))

    def get_workload(self, hour: int = 0) -> int:
        """
        Lấy workload theo pattern đã cấu hình.

        Args:
            hour: Giờ trong ngày (dùng cho periodic pattern)

        Returns:
            Số jobs cần xử lý
        """
        if self.pattern == "stable":
            return self._get_stable_workload()
        elif self.pattern == "spike":
            return self._get_spike_workload()
        elif self.pattern == "random":
            return self._get_random_workload()
        elif self.pattern == "periodic":
            return self._get_periodic_workload(hour)
        else:
            return self._get_stable_workload()

    def step(self, hour: int = 0) -> int:
        """
        Tiến một step và trả về workload.

        Args:
            hour: Giờ trong ngày

        Returns:
            Số jobs mới đến
        """
        workload = self.get_workload(hour)
        self.current_step += 1
        return workload

    def generate_workload_series(self, n_steps: int,
                                  start_hour: int = 0) -> np.ndarray:
        """
        Sinh chuỗi workload cho n_steps.

        Args:
            n_steps: Số steps cần sinh
            start_hour: Giờ bắt đầu

        Returns:
            Array workload cho mỗi step
        """
        workloads = np.zeros(n_steps, dtype=int)

        for i in range(n_steps):
            hour = (start_hour + i) % 24
            workloads[i] = self.get_workload(hour)

        return workloads


In [ ]:
def generate_sample_data(n_days: int = 7, seed: int = 42) -> pd.DataFrame:
    """
    Sinh dữ liệu mẫu bao gồm giá spot và workload.

    Args:
        n_days: Số ngày dữ liệu
        seed: Random seed

    Returns:
        DataFrame chứa timestamp, spot_price, workload
    """
    n_steps = n_days * 24  # Mỗi step là 1 giờ

    # Khởi tạo generators
    price_gen = SpotPriceGenerator(seed=seed)
    workload_gen = WorkloadGenerator(pattern="spike", seed=seed)

    # Sinh dữ liệu
    data = []
    for i in range(n_steps):
        price, hour, day = price_gen.step()
        workload = workload_gen.step(hour)

        data.append({
            'step': i,
            'hour': hour,
            'day_of_week': day,
            'spot_price': round(price, 4),
            'workload': workload
        })

    return pd.DataFrame(data)


def save_sample_data(filepath: str, n_days: int = 7, seed: int = 42) -> None:
    """
    Sinh và lưu dữ liệu mẫu vào file CSV.

    Args:
        filepath: Đường dẫn file output
        n_days: Số ngày dữ liệu
        seed: Random seed
    """
    df = generate_sample_data(n_days, seed)
    df.to_csv(filepath, index=False)
    print(f"Đã lưu dữ liệu mẫu vào {filepath}")
    print(f"Số dòng: {len(df)}")
    print(f"Các cột: {list(df.columns)}")


if __name__ == "__main__":
    # Demo: Sinh và hiển thị dữ liệu mẫu
    print("=" * 60)
    print("Demo: Spot Price Generator")
    print("=" * 60)

    price_gen = SpotPriceGenerator()
    prices = price_gen.generate_price_series(24)  # 1 ngày
    print(f"Giá spot trong 24 giờ đầu tiên:")
    print(f"Min: ${min(prices):.4f}, Max: ${max(prices):.4f}, Mean: ${np.mean(prices):.4f}")

    print("\n" + "=" * 60)
    print("Demo: Workload Generator")
    print("=" * 60)

    for pattern in ["stable", "spike", "random", "periodic"]:
        wl_gen = WorkloadGenerator(pattern=pattern)
        workloads = wl_gen.generate_workload_series(24)
        print(f"\nPattern '{pattern}':")
        print(f"Min: {min(workloads)}, Max: {max(workloads)}, Mean: {np.mean(workloads):.1f}")

    print("\n" + "=" * 60)
    print("Demo: Sinh dữ liệu mẫu")
    print("=" * 60)

    df = generate_sample_data(n_days=1)
    print(df.head(10))


In [ ]:
# -*- coding: utf-8 -*-
"""
Gymnasium Environment cho Spot Instance Bidding Strategy

Môi trường này mô phỏng việc quản lý cloud instances với 2 loại:
- Spot Instance: Rẻ nhưng có thể bị gián đoạn bất kỳ lúc nào
- On-Demand Instance: Đắt hơn nhưng ổn định

Agent cần học cách tối ưu việc sử dụng các loại instance này
để giảm chi phí trong khi vẫn đảm bảo hoàn thành workload.
"""

import gymnasium as gym
from gymnasium import spaces
import numpy as np
from typing import Optional, Dict, Any, Tuple, List
from dataclasses import dataclass, field
from collections import deque



@dataclass


In [ ]:
class InstanceConfig:
    """Cấu hình cho môi trường Spot Instance"""
    # Giá instance
    on_demand_price: float = 0.10          # Giá on-demand ($/hour)
    spot_price_base: float = 0.03          # Giá spot cơ bản
    spot_price_max: float = 0.08           # Giá spot tối đa

    # Giới hạn instances
    max_instances: int = 10                 # Số instance tối đa
    max_pending_jobs: int = 100             # Số jobs tối đa trong queue

    # Hiệu suất xử lý
    jobs_per_instance_per_step: int = 2     # Jobs mỗi instance xử lý/step

    # Xác suất interruption
    base_interruption_prob: float = 0.05    # Xác suất bị terminate cơ bản
    high_price_interruption_prob: float = 0.15  # Xác suất khi giá cao

    # Thời gian mô phỏng
    episode_length: int = 168               # Số steps/episode (168 = 1 tuần)

    # Checkpointing
    checkpoint_cost: float = 0.01           # Chi phí checkpoint
    checkpoint_recovery_rate: float = 0.8   # Tỷ lệ recovery sau checkpoint


@dataclass
class RewardConfig:
    """Cấu hình reward function"""
    alpha: float = 1.0      # Hệ số phạt chi phí
    beta: float = 0.5       # Hệ số thưởng hoàn thành job
    gamma: float = 2.0      # Hệ số phạt interruption
    delta: float = 1.5      # Hệ số phạt vi phạm SLA

    # Ngưỡng SLA
    sla_max_pending_jobs: int = 50
    sla_max_wait_time: int = 5


@dataclass


In [ ]:
class Instance:
    """Đại diện cho một cloud instance"""
    instance_type: str              # "spot" hoặc "on_demand"
    created_at: int                 # Step được tạo
    jobs_processed: int = 0         # Số jobs đã xử lý
    is_checkpointed: bool = False   # Đã checkpoint chưa


In [ ]:
class SpotInstanceEnv(gym.Env):
    """
    Gymnasium Environment cho bài toán Spot Instance Bidding.

    State Space (7 features):
        - current_spot_price: Giá spot hiện tại (0 đến max_price)
        - price_moving_avg: Giá trung bình 1 giờ qua
        - hour_of_day: Giờ trong ngày (0-23), normalized to [0,1]
        - day_of_week: Ngày trong tuần (0-6), normalized to [0,1]
        - pending_workload: Số jobs cần xử lý (normalized)
        - active_instances: Số instances đang chạy (normalized)
        - interruption_probability: Xác suất bị terminate (0-1)

    Action Space (5 discrete actions):
        - 0: do_nothing - Không làm gì
        - 1: request_spot_instance - Yêu cầu spot instance mới
        - 2: request_on_demand_instance - Yêu cầu on-demand instance mới
        - 3: terminate_instance - Terminate 1 instance (ưu tiên on-demand trước)
        - 4: checkpoint_and_migrate - Checkpoint spot instances và migrate

    Reward:
        R = -α(cost) + β(jobs_completed) - γ(interruption_penalty) - δ(sla_violation)
    """

    # Metadata cho Gymnasium
    metadata = {'render_modes': ['human', 'ansi']}

    # Định nghĩa các action
    ACTION_DO_NOTHING = 0
    ACTION_REQUEST_SPOT = 1
    ACTION_REQUEST_ON_DEMAND = 2
    ACTION_TERMINATE = 3
    ACTION_CHECKPOINT_MIGRATE = 4

    ACTION_NAMES = {
        0: "do_nothing",
        1: "request_spot",
        2: "request_on_demand",
        3: "terminate",
        4: "checkpoint_migrate"
    }

    def __init__(
        self,
        instance_config: Optional[InstanceConfig] = None,
        reward_config: Optional[RewardConfig] = None,
        workload_pattern: str = "spike",
        seed: Optional[int] = None,
        render_mode: Optional[str] = None
    ):
        """
        Khởi tạo môi trường.

        Args:
            instance_config: Cấu hình instance
            reward_config: Cấu hình reward
            workload_pattern: Pattern của workload ("stable", "spike", "random", "periodic")
            seed: Random seed
            render_mode: Chế độ render ("human", "ansi", None)
        """
        super().__init__()

        self.instance_config = instance_config or InstanceConfig()
        self.reward_config = reward_config or RewardConfig()
        self.workload_pattern = workload_pattern
        self.render_mode = render_mode

        # Định nghĩa action space (5 discrete actions)
        self.action_space = spaces.Discrete(5)

        # Định nghĩa observation space (7 continuous features)
        # Tất cả được normalize về khoảng [0, 1]
        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(7,),
            dtype=np.float32
        )

        # Khởi tạo các generator
        price_config = PriceConfig(
            base_price=self.instance_config.spot_price_base,
            max_price=self.instance_config.spot_price_max
        )
        workload_config = WorkloadConfig(
            max_workload=self.instance_config.max_pending_jobs
        )

        self._seed = seed
        self.price_generator = SpotPriceGenerator(config=price_config, seed=seed or 42)
        self.workload_generator = WorkloadGenerator(
            config=workload_config,
            pattern=workload_pattern,
            seed=seed or 42
        )

        # Khởi tạo price history cho moving average
        self.price_history: deque = deque(maxlen=24)  # 24 giờ

        # Trạng thái môi trường
        self.instances: List[Instance] = []
        self.pending_jobs: int = 0
        self.current_step: int = 0
        self.current_hour: int = 0
        self.current_day: int = 0
        self.current_spot_price: float = 0.0

        # Metrics để theo dõi
        self.total_cost: float = 0.0
        self.total_jobs_completed: int = 0
        self.total_interruptions: int = 0
        self.total_sla_violations: int = 0

        # History để phân tích
        self.history: List[Dict[str, Any]] = []

    def reset(
        self,
        seed: Optional[int] = None,
        options: Optional[Dict[str, Any]] = None
    ) -> Tuple[np.ndarray, Dict[str, Any]]:
        """
        Reset môi trường về trạng thái ban đầu.

        Args:
            seed: Random seed mới (optional)
            options: Options bổ sung (optional)

        Returns:
            Tuple (observation, info)
        """
        super().reset(seed=seed)

        if seed is not None:
            self._seed = seed
            self.price_generator = SpotPriceGenerator(seed=seed)
            self.workload_generator = WorkloadGenerator(
                pattern=self.workload_pattern,
                seed=seed
            )

        # Reset generators
        self.price_generator.reset()
        self.workload_generator.reset()

        # Reset trạng thái
        self.instances = []
        self.pending_jobs = 0
        self.current_step = 0
        self.price_history.clear()

        # Reset metrics
        self.total_cost = 0.0
        self.total_jobs_completed = 0
        self.total_interruptions = 0
        self.total_sla_violations = 0
        self.history = []

        # Lấy giá spot ban đầu
        self.current_spot_price, self.current_hour, self.current_day = \
            self.price_generator.step()
        self.price_history.append(self.current_spot_price)

        # Thêm workload ban đầu
        initial_workload = self.workload_generator.step(self.current_hour)
        self.pending_jobs = min(initial_workload, self.instance_config.max_pending_jobs)

        # Tạo observation
        obs = self._get_observation()

        # Thông tin bổ sung
        info = {
            'spot_price': self.current_spot_price,
            'hour': self.current_hour,
            'day': self.current_day,
            'pending_jobs': self.pending_jobs,
            'n_instances': len(self.instances)
        }

        return obs, info

    def _get_observation(self) -> np.ndarray:
        """
        Tạo observation từ trạng thái hiện tại.

        Returns:
            Array 7 features, normalized về [0, 1]
        """
        # 1. Current spot price (normalized)
        price_normalized = self.current_spot_price / self.instance_config.spot_price_max

        # 2. Price moving average (normalized)
        if len(self.price_history) > 0:
            price_avg = np.mean(self.price_history)
        else:
            price_avg = self.current_spot_price
        price_avg_normalized = price_avg / self.instance_config.spot_price_max

        # 3. Hour of day (normalized to [0, 1])
        hour_normalized = self.current_hour / 23.0

        # 4. Day of week (normalized to [0, 1])
        day_normalized = self.current_day / 6.0

        # 5. Pending workload (normalized)
        pending_normalized = self.pending_jobs / self.instance_config.max_pending_jobs

        # 6. Active instances (normalized)
        instances_normalized = len(self.instances) / self.instance_config.max_instances

        # 7. Interruption probability
        interruption_prob = self.price_generator.get_interruption_probability(
            self.current_spot_price
        )

        obs = np.array([
            price_normalized,
            price_avg_normalized,
            hour_normalized,
            day_normalized,
            pending_normalized,
            instances_normalized,
            interruption_prob
        ], dtype=np.float32)

        # Clip về [0, 1] để đảm bảo
        obs = np.clip(obs, 0.0, 1.0)

        return obs

    def step(self, action: int) -> Tuple[np.ndarray, float, bool, bool, Dict[str, Any]]:
        """
        Thực hiện một action và tiến tới step tiếp theo.

        Args:
            action: Action để thực hiện (0-4)

        Returns:
            Tuple (observation, reward, terminated, truncated, info)
        """
        # Validate action
        assert self.action_space.contains(action), f"Invalid action: {action}"

        # Lưu trạng thái trước khi action
        step_info = {
            'step': self.current_step,
            'action': action,
            'action_name': self.ACTION_NAMES[action],
            'spot_price': self.current_spot_price,
            'pending_jobs_before': self.pending_jobs,
            'n_instances_before': len(self.instances),
            'n_spot_before': self._count_spot_instances(),
            'n_on_demand_before': self._count_on_demand_instances()
        }

        # Thực hiện action
        action_cost, action_success = self._execute_action(action)

        # Xử lý interruptions cho spot instances
        interruption_penalty, n_interrupted = self._process_interruptions()

        # Xử lý jobs
        jobs_completed = self._process_jobs()

        # Thêm workload mới
        new_workload = self.workload_generator.step(self.current_hour)
        self.pending_jobs = min(
            self.pending_jobs + new_workload,
            self.instance_config.max_pending_jobs
        )

        # Tính chi phí running
        running_cost = self._calculate_running_cost()

        # Tính SLA violation
        sla_violation = self._check_sla_violation()
        if sla_violation > 0:
            self.total_sla_violations += 1

        # Tính reward
        reward = self._calculate_reward(
            action_cost + running_cost,
            jobs_completed,
            interruption_penalty,
            sla_violation
        )

        # Cập nhật metrics
        self.total_cost += action_cost + running_cost
        self.total_jobs_completed += jobs_completed
        self.total_interruptions += n_interrupted

        # Tiến tới step tiếp theo
        self.current_step += 1
        self.current_spot_price, self.current_hour, self.current_day = \
            self.price_generator.step()
        self.price_history.append(self.current_spot_price)

        # Kiểm tra kết thúc episode
        terminated = False  # Không có điều kiện kết thúc sớm
        truncated = self.current_step >= self.instance_config.episode_length

        # Tạo observation mới
        obs = self._get_observation()

        # Cập nhật step info
        step_info.update({
            'action_success': action_success,
            'action_cost': action_cost,
            'running_cost': running_cost,
            'jobs_completed': jobs_completed,
            'n_interrupted': n_interrupted,
            'interruption_penalty': interruption_penalty,
            'sla_violation': sla_violation,
            'reward': reward,
            'pending_jobs_after': self.pending_jobs,
            'n_instances_after': len(self.instances),
            'n_spot_after': self._count_spot_instances(),
            'n_on_demand_after': self._count_on_demand_instances()
        })
        self.history.append(step_info)

        # Info trả về
        info = {
            'step': self.current_step,
            'spot_price': self.current_spot_price,
            'hour': self.current_hour,
            'day': self.current_day,
            'pending_jobs': self.pending_jobs,
            'n_instances': len(self.instances),
            'n_spot': self._count_spot_instances(),
            'n_on_demand': self._count_on_demand_instances(),
            'jobs_completed': jobs_completed,
            'cost': action_cost + running_cost,
            'total_cost': self.total_cost,
            'total_jobs': self.total_jobs_completed,
            'total_interruptions': self.total_interruptions
        }

        return obs, reward, terminated, truncated, info

    def _execute_action(self, action: int) -> Tuple[float, bool]:
        """
        Thực hiện action và trả về chi phí và trạng thái thành công.

        Args:
            action: Action cần thực hiện

        Returns:
            Tuple (cost, success)
        """
        cost = 0.0
        success = True

        if action == self.ACTION_DO_NOTHING:
            # Không làm gì
            pass

        elif action == self.ACTION_REQUEST_SPOT:
            # Yêu cầu spot instance mới
            if len(self.instances) < self.instance_config.max_instances:
                new_instance = Instance(
                    instance_type="spot",
                    created_at=self.current_step
                )
                self.instances.append(new_instance)
            else:
                success = False

        elif action == self.ACTION_REQUEST_ON_DEMAND:
            # Yêu cầu on-demand instance mới
            if len(self.instances) < self.instance_config.max_instances:
                new_instance = Instance(
                    instance_type="on_demand",
                    created_at=self.current_step
                )
                self.instances.append(new_instance)
            else:
                success = False

        elif action == self.ACTION_TERMINATE:
            # Terminate instance (ưu tiên on-demand trước để tiết kiệm chi phí)
            if len(self.instances) > 0:
                # Tìm on-demand instance trước
                on_demand_idx = None
                for i, inst in enumerate(self.instances):
                    if inst.instance_type == "on_demand":
                        on_demand_idx = i
                        break

                if on_demand_idx is not None:
                    self.instances.pop(on_demand_idx)
                else:
                    # Nếu không có on-demand, terminate spot instance
                    self.instances.pop(0)
            else:
                success = False

        elif action == self.ACTION_CHECKPOINT_MIGRATE:
            # Checkpoint tất cả spot instances
            spot_count = self._count_spot_instances()
            if spot_count > 0:
                cost = spot_count * self.instance_config.checkpoint_cost
                for inst in self.instances:
                    if inst.instance_type == "spot":
                        inst.is_checkpointed = True
            else:
                success = False

        return cost, success

    def _process_interruptions(self) -> Tuple[float, int]:
        """
        Xử lý interruptions cho spot instances.

        Returns:
            Tuple (total_penalty, n_interrupted)
        """
        total_penalty = 0.0
        n_interrupted = 0
        interruption_prob = self.price_generator.get_interruption_probability(
            self.current_spot_price
        )

        # Lọc ra các spot instances bị interrupt
        surviving_instances = []
        for inst in self.instances:
            if inst.instance_type == "spot":
                if np.random.random() < interruption_prob:
                    # Instance bị interrupt
                    n_interrupted += 1

                    if inst.is_checkpointed:
                        # Có checkpoint - penalty nhẹ hơn
                        total_penalty += 0.5
                        # Tạo instance mới với một phần jobs được recover
                        recovered_jobs = int(inst.jobs_processed *
                                           self.instance_config.checkpoint_recovery_rate)
                        # Reset checkpoint flag
                        inst.is_checkpointed = False
                        inst.jobs_processed = recovered_jobs
                        surviving_instances.append(inst)
                    else:
                        # Không có checkpoint - penalty full
                        total_penalty += 1.0
                else:
                    surviving_instances.append(inst)
            else:
                # On-demand không bị interrupt
                surviving_instances.append(inst)

        self.instances = surviving_instances
        return total_penalty, n_interrupted

    def _process_jobs(self) -> int:
        """
        Xử lý jobs với các instances hiện có.

        Returns:
            Số jobs đã hoàn thành
        """
        if self.pending_jobs == 0 or len(self.instances) == 0:
            return 0

        # Tính tổng capacity
        total_capacity = len(self.instances) * self.instance_config.jobs_per_instance_per_step

        # Số jobs được xử lý
        jobs_completed = min(self.pending_jobs, total_capacity)

        # Cập nhật pending jobs
        self.pending_jobs -= jobs_completed

        # Cập nhật jobs processed cho mỗi instance
        jobs_per_instance = jobs_completed // max(1, len(self.instances))
        for inst in self.instances:
            inst.jobs_processed += jobs_per_instance

        return jobs_completed

    def _calculate_running_cost(self) -> float:
        """
        Tính chi phí running của tất cả instances.

        Returns:
            Tổng chi phí trong step này
        """
        total_cost = 0.0
        for inst in self.instances:
            if inst.instance_type == "spot":
                total_cost += self.current_spot_price
            else:
                total_cost += self.instance_config.on_demand_price
        return total_cost

    def _check_sla_violation(self) -> float:
        """
        Kiểm tra vi phạm SLA.

        Returns:
            Giá trị vi phạm (0 nếu không vi phạm)
        """
        violation = 0.0

        # Vi phạm nếu pending jobs vượt ngưỡng
        if self.pending_jobs > self.reward_config.sla_max_pending_jobs:
            excess_ratio = (self.pending_jobs - self.reward_config.sla_max_pending_jobs) / \
                          self.reward_config.sla_max_pending_jobs
            violation = min(1.0, excess_ratio)

        return violation

    def _calculate_reward(
        self,
        cost: float,
        jobs_completed: int,
        interruption_penalty: float,
        sla_violation: float
    ) -> float:
        """
        Tính reward theo công thức:
        R = -α(cost) + β(jobs_completed) - γ(interruption_penalty) - δ(sla_violation)

        Args:
            cost: Chi phí trong step này
            jobs_completed: Số jobs hoàn thành
            interruption_penalty: Penalty từ interruptions
            sla_violation: Mức độ vi phạm SLA

        Returns:
            Giá trị reward
        """
        reward = (
            -self.reward_config.alpha * cost
            + self.reward_config.beta * jobs_completed
            - self.reward_config.gamma * interruption_penalty
            - self.reward_config.delta * sla_violation
        )
        return reward

    def _count_spot_instances(self) -> int:
        """Đếm số spot instances"""
        return sum(1 for inst in self.instances if inst.instance_type == "spot")

    def _count_on_demand_instances(self) -> int:
        """Đếm số on-demand instances"""
        return sum(1 for inst in self.instances if inst.instance_type == "on_demand")

    def get_metrics(self) -> Dict[str, Any]:
        """
        Lấy các metrics của episode hiện tại.

        Returns:
            Dictionary chứa các metrics
        """
        cost_per_job = self.total_cost / max(1, self.total_jobs_completed)

        return {
            'total_cost': self.total_cost,
            'total_jobs_completed': self.total_jobs_completed,
            'total_interruptions': self.total_interruptions,
            'total_sla_violations': self.total_sla_violations,
            'cost_per_job': cost_per_job,
            'n_steps': self.current_step,
            'final_pending_jobs': self.pending_jobs,
            'final_n_instances': len(self.instances)
        }

    def get_history(self) -> List[Dict[str, Any]]:
        """
        Lấy lịch sử của episode.

        Returns:
            List các step info
        """
        return self.history.copy()

    def render(self) -> Optional[str]:
        """
        Render trạng thái hiện tại của môi trường.

        Returns:
            String mô tả trạng thái (nếu render_mode='ansi')
        """
        output = []
        output.append(f"\n{'='*60}")
        output.append(f"Step: {self.current_step} | Hour: {self.current_hour} | Day: {self.current_day}")
        output.append(f"{'='*60}")
        output.append(f"Spot Price: ${self.current_spot_price:.4f}")
        output.append(f"Pending Jobs: {self.pending_jobs}")
        output.append(f"Instances: {len(self.instances)} "
                     f"(Spot: {self._count_spot_instances()}, "
                     f"On-Demand: {self._count_on_demand_instances()})")
        output.append(f"Total Cost: ${self.total_cost:.4f}")
        output.append(f"Total Jobs Completed: {self.total_jobs_completed}")
        output.append(f"Total Interruptions: {self.total_interruptions}")
        output.append(f"{'='*60}")

        render_str = '\n'.join(output)

        if self.render_mode == 'human':
            print(render_str)
        elif self.render_mode == 'ansi':
            return render_str

        return None

    def close(self):
        """Đóng môi trường và giải phóng tài nguyên"""
        pass


# Đăng ký environment với Gymnasium


In [ ]:
def register_env():
    """Đăng ký SpotInstanceEnv với Gymnasium registry"""
    try:
        gym.register(
            id='SpotInstance-v0',
            entry_point='src.environment:SpotInstanceEnv',
            max_episode_steps=168,
        )
    except gym.error.Error:
        # Đã được đăng ký rồi
        pass


if __name__ == "__main__":
    # Demo: Test environment
    print("=" * 60)
    print("Demo: SpotInstanceEnv")
    print("=" * 60)

    env = SpotInstanceEnv(render_mode='human')
    obs, info = env.reset(seed=42)

    print(f"\nInitial observation: {obs}")
    print(f"Observation shape: {obs.shape}")

    # Chạy một vài steps
    total_reward = 0
    for i in range(10):
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward

        print(f"\nStep {i+1}:")
        print(f"  Action: {env.ACTION_NAMES[action]}")
        print(f"  Reward: {reward:.4f}")
        env.render()

        if terminated or truncated:
            break

    print(f"\n{'='*60}")
    print("Episode Metrics:")
    print("=" * 60)
    metrics = env.get_metrics()
    for key, value in metrics.items():
        print(f"  {key}: {value}")

    env.close()


## PART 1: State Features and Normalization


In [ ]:
### YOU NEED TO WRITE YOUR CODE BELOW ###
def normalize_obs(obs):
    return None


## PART 2: Heuristic Policies


In [ ]:
### YOU NEED TO WRITE YOUR CODE BELOW ###
class SmartThresholdPolicy:
    def predict(self, obs):
        return 0


## PART 3: Deep Q-Network Implementation


In [ ]:
### YOU NEED TO WRITE YOUR CODE BELOW ###
class DQNNetwork(nn.Module):
    def __init__(self, s=7, a=5):
        pass


In [ ]:
### YOU NEED TO WRITE YOUR CODE BELOW ###
def compute_dqn_loss(model, target, batch):
    return torch.tensor(0.0)


## PART 4: Evaluation and Visualization


In [ ]:
# -*- coding: utf-8 -*-
"""
Visualization Module cho Spot Instance Bidding Strategy

Module này cung cấp các hàm để visualization:
- Reward curves từ training
- Cost comparison bar charts
- Action distribution
- Price vs Action timeline
- Episode analysis

Sử dụng:
    from src.visualize import plot_training_curves, plot_episode_analysis
"""

import os
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec


# Cấu hình style chung
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {
    'spot': '#2ca02c',
    'on_demand': '#1f77b4',
    'cost': '#d62728',
    'reward': '#ff7f0e',
    'jobs': '#9467bd',
    'interruption': '#e377c2'
}

ACTION_COLORS = {
    0: '#7f7f7f',  # Do Nothing - Gray
    1: '#2ca02c',  # Request Spot - Green
    2: '#1f77b4',  # Request On-Demand - Blue
    3: '#d62728',  # Terminate - Red
    4: '#ff7f0e'   # Checkpoint - Orange
}

ACTION_NAMES = {
    0: 'Do Nothing',
    1: 'Request Spot',
    2: 'Request On-Demand',
    3: 'Terminate',
    4: 'Checkpoint'
}


In [ ]:
def plot_training_curves(
    log_path: str,
    save_path: Optional[str] = None,
    show: bool = True
) -> plt.Figure:
    """
    Vẽ training curves từ TensorBoard logs hoặc CSV.

    Args:
        log_path: Đường dẫn đến log files
        save_path: Đường dẫn lưu plot (optional)
        show: Hiển thị plot

    Returns:
        Figure object
    """
    # Tìm và đọc các file log
    log_dir = Path(log_path)

    # Tìm file monitor.csv nếu có
    monitor_files = list(log_dir.glob("**/monitor.csv"))

    if not monitor_files:
        print(f"No monitor files found in {log_path}")
        return None

    # Đọc và combine data
    all_data = []
    for f in monitor_files:
        try:
            df = pd.read_csv(f, skiprows=1)
            all_data.append(df)
        except Exception as e:
            print(f"Error reading {f}: {e}")

    if not all_data:
        print("No data to plot")
        return None

    data = pd.concat(all_data, ignore_index=True)

    # Tạo figure với subplots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Training Curves - Spot Instance Bidding Strategy', fontsize=14)

    # 1. Episode Reward
    ax1 = axes[0, 0]
    rewards = data['r'].values
    episodes = np.arange(len(rewards))

    # Rolling mean
    window = min(100, len(rewards) // 10)
    if window > 0:
        rolling_mean = pd.Series(rewards).rolling(window=window).mean()
        ax1.plot(episodes, rewards, alpha=0.3, color=COLORS['reward'], label='Episode Reward')
        ax1.plot(episodes, rolling_mean, color=COLORS['reward'], linewidth=2,
                 label=f'Rolling Mean ({window})')
    else:
        ax1.plot(episodes, rewards, color=COLORS['reward'])

    ax1.set_xlabel('Episode')
    ax1.set_ylabel('Reward')
    ax1.set_title('Episode Reward over Training')
    ax1.legend()

    # 2. Episode Length
    ax2 = axes[0, 1]
    lengths = data['l'].values
    ax2.plot(episodes, lengths, alpha=0.5, color='#17becf')
    ax2.set_xlabel('Episode')
    ax2.set_ylabel('Steps')
    ax2.set_title('Episode Length')
    ax2.axhline(y=168, color='r', linestyle='--', label='Expected (168)')
    ax2.legend()

    # 3. Cumulative Reward
    ax3 = axes[1, 0]
    cumsum_rewards = np.cumsum(rewards)
    ax3.plot(episodes, cumsum_rewards, color=COLORS['reward'])
    ax3.set_xlabel('Episode')
    ax3.set_ylabel('Cumulative Reward')
    ax3.set_title('Cumulative Reward')

    # 4. Reward Distribution
    ax4 = axes[1, 1]
    ax4.hist(rewards, bins=50, color=COLORS['reward'], alpha=0.7, edgecolor='black')
    ax4.axvline(x=np.mean(rewards), color='r', linestyle='--',
                label=f'Mean: {np.mean(rewards):.2f}')
    ax4.set_xlabel('Reward')
    ax4.set_ylabel('Frequency')
    ax4.set_title('Reward Distribution')
    ax4.legend()

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Plot saved to: {save_path}")

    if show:
        plt.show()

    return fig


In [ ]:
def plot_episode_analysis(
    history: List[Dict[str, Any]],
    save_path: Optional[str] = None,
    show: bool = True,
    title: str = "Episode Analysis"
) -> plt.Figure:
    """
    Phân tích chi tiết một episode.

    Args:
        history: List các step info từ env.get_history()
        save_path: Đường dẫn lưu plot
        show: Hiển thị plot
        title: Tiêu đề

    Returns:
        Figure object
    """
    # Convert to DataFrame
    df = pd.DataFrame(history)

    fig = plt.figure(figsize=(16, 12))
    gs = GridSpec(3, 3, figure=fig)

    fig.suptitle(f'{title}', fontsize=14)

    # 1. Spot Price Timeline (top row, full width)
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(df['step'], df['spot_price'], color=COLORS['cost'], linewidth=1.5)
    ax1.fill_between(df['step'], df['spot_price'], alpha=0.3, color=COLORS['cost'])
    ax1.set_ylabel('Spot Price ($)')
    ax1.set_title('Giá Spot theo thời gian')

    # Highlight các action
    for action_id in [1, 2]:  # Request spot, Request on-demand
        action_steps = df[df['action'] == action_id]['step']
        action_prices = df[df['action'] == action_id]['spot_price']
        ax1.scatter(action_steps, action_prices, c=ACTION_COLORS[action_id],
                   s=50, zorder=5, label=ACTION_NAMES[action_id])
    ax1.legend(loc='upper right')

    # 2. Actions Timeline
    ax2 = fig.add_subplot(gs[1, :], sharex=ax1)
    scatter_colors = [ACTION_COLORS[a] for a in df['action']]
    ax2.scatter(df['step'], df['action'], c=scatter_colors, s=30, alpha=0.7)
    ax2.set_ylabel('Action')
    ax2.set_yticks([0, 1, 2, 3, 4])
    ax2.set_yticklabels(list(ACTION_NAMES.values()), fontsize=8)
    ax2.set_title('Actions theo thời gian')

    # 3. Instance Count
    ax3 = fig.add_subplot(gs[2, 0])
    ax3.plot(df['step'], df['n_spot_after'], label='Spot', color=COLORS['spot'], linewidth=2)
    ax3.plot(df['step'], df['n_on_demand_after'], label='On-Demand',
             color=COLORS['on_demand'], linewidth=2)
    ax3.plot(df['step'], df['n_instances_after'], label='Total',
             color='black', linestyle='--', linewidth=1)
    ax3.set_xlabel('Step')
    ax3.set_ylabel('Count')
    ax3.set_title('Số Instances')
    ax3.legend()

    # 4. Pending Jobs
    ax4 = fig.add_subplot(gs[2, 1])
    ax4.plot(df['step'], df['pending_jobs_before'], color=COLORS['jobs'], linewidth=1.5)
    ax4.fill_between(df['step'], df['pending_jobs_before'], alpha=0.3, color=COLORS['jobs'])
    ax4.set_xlabel('Step')
    ax4.set_ylabel('Jobs')
    ax4.set_title('Pending Jobs')
    ax4.axhline(y=50, color='r', linestyle='--', alpha=0.5, label='SLA Threshold')
    ax4.legend()

    # 5. Cumulative Metrics
    ax5 = fig.add_subplot(gs[2, 2])
    cumsum_cost = np.cumsum(df['action_cost'] + df['running_cost'])
    cumsum_jobs = np.cumsum(df['jobs_completed'])

    ax5_twin = ax5.twinx()

    line1, = ax5.plot(df['step'], cumsum_cost, color=COLORS['cost'],
                      linewidth=2, label='Cumulative Cost')
    line2, = ax5_twin.plot(df['step'], cumsum_jobs, color=COLORS['jobs'],
                           linewidth=2, label='Cumulative Jobs')

    ax5.set_xlabel('Step')
    ax5.set_ylabel('Cost ($)', color=COLORS['cost'])
    ax5_twin.set_ylabel('Jobs', color=COLORS['jobs'])
    ax5.set_title('Tích lũy Cost & Jobs')
    ax5.legend(handles=[line1, line2], loc='upper left')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Plot saved to: {save_path}")

    if show:
        plt.show()

    return fig


def plot_price_action_timeline(
    history: List[Dict[str, Any]],
    save_path: Optional[str] = None,
    show: bool = True
) -> plt.Figure:
    """
    Vẽ timeline giá spot và actions.

    Args:
        history: Episode history
        save_path: Đường dẫn lưu
        show: Hiển thị

    Returns:
        Figure
    """
    df = pd.DataFrame(history)

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    fig.suptitle('Price vs Action Timeline', fontsize=14)

    # Spot price
    ax1.plot(df['step'], df['spot_price'], color=COLORS['cost'], linewidth=1.5)
    ax1.fill_between(df['step'], df['spot_price'], alpha=0.2, color=COLORS['cost'])

    # Mark interruptions
    interrupted_steps = df[df['n_interrupted'] > 0]['step']
    if len(interrupted_steps) > 0:
        for step in interrupted_steps:
            ax1.axvline(x=step, color=COLORS['interruption'], alpha=0.5, linestyle='--')

    ax1.set_ylabel('Spot Price ($)')
    ax1.set_title('Giá Spot Instance')

    # Actions as colored bars
    for i, row in df.iterrows():
        ax2.bar(row['step'], 1, color=ACTION_COLORS[row['action']], width=1.0)

    ax2.set_xlabel('Step (Hour)')
    ax2.set_ylabel('Action')
    ax2.set_yticks([])

    # Legend
    patches = [mpatches.Patch(color=ACTION_COLORS[i], label=ACTION_NAMES[i])
               for i in range(5)]
    ax2.legend(handles=patches, loc='upper right', ncol=3)
    ax2.set_title('Actions Timeline')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    if show:
        plt.show()

    return fig


def plot_action_distribution(
    results: List[Dict[str, Any]],
    save_path: Optional[str] = None,
    show: bool = True
) -> plt.Figure:
    """
    Vẽ phân phối actions cho nhiều policies.

    Args:
        results: List kết quả từ evaluate
        save_path: Đường dẫn lưu
        show: Hiển thị

    Returns:
        Figure
    """
    fig, ax = plt.subplots(figsize=(12, 6))

    policies = [r['policy_name'] for r in results]
    x = np.arange(len(policies))
    width = 0.6

    bottom = np.zeros(len(policies))

    for action_id in range(5):
        probs = [r['action_distribution'][action_id] for r in results]
        ax.bar(x, probs, width, label=ACTION_NAMES[action_id],
               bottom=bottom, color=ACTION_COLORS[action_id])
        bottom += probs

    ax.set_ylabel('Tỷ lệ Action')
    ax.set_title('Phân phối Actions của các Policy')
    ax.set_xticks(x)
    ax.set_xticklabels(policies, rotation=45, ha='right')
    ax.legend(loc='upper right', bbox_to_anchor=(1.15, 1))

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    if show:
        plt.show()

    return fig


def plot_cost_breakdown(
    history: List[Dict[str, Any]],
    save_path: Optional[str] = None,
    show: bool = True
) -> plt.Figure:
    """
    Phân tích chi tiết chi phí.

    Args:
        history: Episode history
        save_path: Đường dẫn lưu
        show: Hiển thị

    Returns:
        Figure
    """
    df = pd.DataFrame(history)

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle('Chi phí chi tiết trong Episode', fontsize=14)

    # 1. Cost per step
    ax1 = axes[0, 0]
    total_cost = df['action_cost'] + df['running_cost']
    ax1.bar(df['step'], df['running_cost'], label='Running Cost',
            color=COLORS['on_demand'], alpha=0.7)
    ax1.bar(df['step'], df['action_cost'], bottom=df['running_cost'],
            label='Action Cost', color=COLORS['cost'], alpha=0.7)
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Cost ($)')
    ax1.set_title('Chi phí mỗi Step')
    ax1.legend()

    # 2. Cumulative cost
    ax2 = axes[0, 1]
    cumsum_running = np.cumsum(df['running_cost'])
    cumsum_action = np.cumsum(df['action_cost'])
    ax2.fill_between(df['step'], cumsum_running, label='Running Cost',
                     color=COLORS['on_demand'], alpha=0.5)
    ax2.fill_between(df['step'], cumsum_running, cumsum_running + cumsum_action,
                     label='Action Cost', color=COLORS['cost'], alpha=0.5)
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Cumulative Cost ($)')
    ax2.set_title('Tích lũy Chi phí')
    ax2.legend()

    # 3. Cost vs Jobs trade-off
    ax3 = axes[1, 0]
    cumsum_cost = np.cumsum(total_cost)
    cumsum_jobs = np.cumsum(df['jobs_completed'])
    ax3.scatter(cumsum_cost, cumsum_jobs, c=df['step'], cmap='viridis', s=20)
    ax3.set_xlabel('Cumulative Cost ($)')
    ax3.set_ylabel('Cumulative Jobs')
    ax3.set_title('Trade-off: Cost vs Jobs')
    cbar = plt.colorbar(ax3.collections[0], ax=ax3, label='Step')

    # 4. Cost efficiency over time
    ax4 = axes[1, 1]
    efficiency = cumsum_jobs / (cumsum_cost + 0.001)  # Jobs per dollar
    ax4.plot(df['step'], efficiency, color=COLORS['jobs'], linewidth=2)
    ax4.set_xlabel('Step')
    ax4.set_ylabel('Jobs per Dollar')
    ax4.set_title('Hiệu quả Chi phí theo thời gian')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    if show:
        plt.show()

    return fig


def plot_spot_price_patterns(
    n_days: int = 7,
    save_path: Optional[str] = None,
    show: bool = True
) -> plt.Figure:
    """
    Visualization patterns giá spot.

    Args:
        n_days: Số ngày để visualize
        save_path: Đường dẫn lưu
        show: Hiển thị

    Returns:
        Figure
    """

    gen = SpotPriceGenerator(seed=42)
    n_steps = n_days * 24
    prices = gen.generate_price_series(n_steps)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Patterns Giá Spot Instance', fontsize=14)

    # 1. Price over time
    ax1 = axes[0, 0]
    hours = np.arange(n_steps)
    ax1.plot(hours, prices, color=COLORS['cost'], linewidth=1)
    ax1.fill_between(hours, prices, alpha=0.3, color=COLORS['cost'])
    ax1.set_xlabel('Hour')
    ax1.set_ylabel('Price ($)')
    ax1.set_title(f'Giá Spot trong {n_days} ngày')

    # Đánh dấu các ngày
    for day in range(1, n_days):
        ax1.axvline(x=day * 24, color='gray', linestyle='--', alpha=0.5)

    # 2. Average by hour of day
    ax2 = axes[0, 1]
    prices_by_hour = [prices[i::24] for i in range(24)]
    mean_by_hour = [np.mean(p) for p in prices_by_hour]
    std_by_hour = [np.std(p) for p in prices_by_hour]

    ax2.bar(range(24), mean_by_hour, yerr=std_by_hour, capsize=3,
            color=COLORS['cost'], alpha=0.7)
    ax2.set_xlabel('Hour of Day')
    ax2.set_ylabel('Average Price ($)')
    ax2.set_title('Giá trung bình theo Giờ')
    ax2.set_xticks(range(0, 24, 2))

    # 3. Average by day of week
    ax3 = axes[1, 0]
    day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
    prices_reshaped = prices[:n_days * 24].reshape(n_days, 24)

    if n_days >= 7:
        prices_by_day = []
        for d in range(7):
            day_indices = [i for i in range(n_days) if i % 7 == d]
            day_prices = [prices_reshaped[i].mean() for i in day_indices if i < n_days]
            prices_by_day.append(np.mean(day_prices) if day_prices else 0)
    else:
        prices_by_day = [prices_reshaped[d % n_days].mean() for d in range(7)]

    ax3.bar(range(7), prices_by_day, color=COLORS['cost'], alpha=0.7)
    ax3.set_xlabel('Day of Week')
    ax3.set_ylabel('Average Price ($)')
    ax3.set_title('Giá trung bình theo Ngày')
    ax3.set_xticks(range(7))
    ax3.set_xticklabels(day_names)

    # 4. Price distribution
    ax4 = axes[1, 1]
    ax4.hist(prices, bins=50, color=COLORS['cost'], alpha=0.7, edgecolor='black')
    ax4.axvline(x=np.mean(prices), color='r', linestyle='--',
                label=f'Mean: ${np.mean(prices):.4f}')
    ax4.axvline(x=np.median(prices), color='b', linestyle='--',
                label=f'Median: ${np.median(prices):.4f}')
    ax4.set_xlabel('Price ($)')
    ax4.set_ylabel('Frequency')
    ax4.set_title('Phân phối Giá')
    ax4.legend()

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    if show:
        plt.show()

    return fig


def plot_workload_patterns(
    n_hours: int = 168,
    save_path: Optional[str] = None,
    show: bool = True
) -> plt.Figure:
    """
    Visualization các pattern workload.

    Args:
        n_hours: Số giờ để visualize
        save_path: Đường dẫn lưu
        show: Hiển thị

    Returns:
        Figure
    """

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Patterns Workload', fontsize=14)

    patterns = ['stable', 'spike', 'random', 'periodic']
    colors = ['#2ca02c', '#d62728', '#9467bd', '#ff7f0e']

    for ax, pattern, color in zip(axes.flat, patterns, colors):
        gen = WorkloadGenerator(pattern=pattern, seed=42)
        workloads = gen.generate_workload_series(n_hours)

        ax.plot(range(n_hours), workloads, color=color, linewidth=1)
        ax.fill_between(range(n_hours), workloads, alpha=0.3, color=color)

        ax.set_xlabel('Hour')
        ax.set_ylabel('Jobs')
        ax.set_title(f'Pattern: {pattern.capitalize()}')

        # Statistics
        stats_text = f'Mean: {np.mean(workloads):.1f}\nMax: {max(workloads)}'
        ax.text(0.95, 0.95, stats_text, transform=ax.transAxes,
                verticalalignment='top', horizontalalignment='right',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')

    if show:
        plt.show()

    return fig


In [ ]:
def create_dashboard(
    history: List[Dict[str, Any]],
    policy_name: str = "Policy",
    save_path: Optional[str] = None,
    show: bool = True
) -> plt.Figure:
    """
    Tạo dashboard tổng hợp cho một episode.

    Args:
        history: Episode history
        policy_name: Tên policy
        save_path: Đường dẫn lưu
        show: Hiển thị

    Returns:
        Figure
    """
    df = pd.DataFrame(history)

    fig = plt.figure(figsize=(18, 12))
    gs = GridSpec(4, 4, figure=fig, hspace=0.3, wspace=0.3)

    fig.suptitle(f'Dashboard: {policy_name}', fontsize=16, fontweight='bold')

    # 1. Spot Price (row 0, cols 0-2)
    ax1 = fig.add_subplot(gs[0, :3])
    ax1.plot(df['step'], df['spot_price'], color=COLORS['cost'], linewidth=1.5)
    ax1.fill_between(df['step'], df['spot_price'], alpha=0.2, color=COLORS['cost'])
    ax1.set_ylabel('Price ($)')
    ax1.set_title('Giá Spot')

    # 2. Summary stats (row 0, col 3)
    ax_stats = fig.add_subplot(gs[0, 3])
    ax_stats.axis('off')

    total_cost = df['action_cost'].sum() + df['running_cost'].sum()
    total_jobs = df['jobs_completed'].sum()
    total_int = df['n_interrupted'].sum()
    cost_per_job = total_cost / max(1, total_jobs)

    stats_text = f"""
    TỔNG KẾT EPISODE

    Tổng Chi phí: ${total_cost:.2f}
    Tổng Jobs: {total_jobs}
    Interruptions: {total_int}
    Cost/Job: ${cost_per_job:.4f}

    Final Instances: {df['n_instances_after'].iloc[-1]}
    - Spot: {df['n_spot_after'].iloc[-1]}
    - On-Demand: {df['n_on_demand_after'].iloc[-1]}
    """
    ax_stats.text(0.1, 0.9, stats_text, transform=ax_stats.transAxes,
                  verticalalignment='top', fontfamily='monospace', fontsize=10,
                  bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.5))

    # 3. Actions (row 1, cols 0-2)
    ax2 = fig.add_subplot(gs[1, :3])
    for i, row in df.iterrows():
        ax2.bar(row['step'], 1, color=ACTION_COLORS[row['action']], width=1.0)
    ax2.set_ylabel('Action')
    ax2.set_yticks([])
    ax2.set_title('Actions Timeline')

    # 4. Action pie chart (row 1, col 3)
    ax_pie = fig.add_subplot(gs[1, 3])
    action_counts = df['action'].value_counts().sort_index()
    colors_pie = [ACTION_COLORS[i] for i in action_counts.index]
    labels_pie = [ACTION_NAMES[i] for i in action_counts.index]
    ax_pie.pie(action_counts.values, labels=labels_pie, colors=colors_pie,
               autopct='%1.1f%%', startangle=90)
    ax_pie.set_title('Action Distribution')

    # 5. Instances (row 2, cols 0-1)
    ax3 = fig.add_subplot(gs[2, :2])
    ax3.stackplot(df['step'],
                  df['n_spot_after'],
                  df['n_on_demand_after'],
                  labels=['Spot', 'On-Demand'],
                  colors=[COLORS['spot'], COLORS['on_demand']],
                  alpha=0.7)
    ax3.set_ylabel('Instances')
    ax3.set_title('Instance Count')
    ax3.legend(loc='upper right')

    # 6. Pending Jobs (row 2, cols 2-3)
    ax4 = fig.add_subplot(gs[2, 2:])
    ax4.plot(df['step'], df['pending_jobs_before'], color=COLORS['jobs'], linewidth=1.5)
    ax4.fill_between(df['step'], df['pending_jobs_before'], alpha=0.3, color=COLORS['jobs'])
    ax4.axhline(y=50, color='r', linestyle='--', alpha=0.5, label='SLA')
    ax4.set_ylabel('Jobs')
    ax4.set_title('Pending Jobs')
    ax4.legend()

    # 7. Cost & Reward (row 3, cols 0-1)
    ax5 = fig.add_subplot(gs[3, :2])
    cumsum_cost = np.cumsum(df['action_cost'] + df['running_cost'])
    cumsum_reward = np.cumsum(df['reward'])

    ax5_twin = ax5.twinx()
    line1, = ax5.plot(df['step'], cumsum_cost, color=COLORS['cost'],
                      linewidth=2, label='Cost')
    line2, = ax5_twin.plot(df['step'], cumsum_reward, color=COLORS['reward'],
                           linewidth=2, label='Reward')

    ax5.set_xlabel('Step')
    ax5.set_ylabel('Cost ($)', color=COLORS['cost'])
    ax5_twin.set_ylabel('Reward', color=COLORS['reward'])
    ax5.set_title('Tích lũy Cost & Reward')
    ax5.legend(handles=[line1, line2])

    # 8. Jobs Completed (row 3, cols 2-3)
    ax6 = fig.add_subplot(gs[3, 2:])
    cumsum_jobs = np.cumsum(df['jobs_completed'])
    ax6.plot(df['step'], cumsum_jobs, color=COLORS['jobs'], linewidth=2)
    ax6.fill_between(df['step'], cumsum_jobs, alpha=0.3, color=COLORS['jobs'])
    ax6.set_xlabel('Step')
    ax6.set_ylabel('Cumulative Jobs')
    ax6.set_title('Jobs Completed')

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Dashboard saved to: {save_path}")

    if show:
        plt.show()

    return fig


if __name__ == "__main__":
    # Demo: Tạo sample visualizations
    print("=" * 60)
    print("Demo: Visualizations")
    print("=" * 60)

    # Plot patterns
    print("\n1. Spot price patterns:")
    plot_spot_price_patterns(n_days=7, show=True)

    print("\n2. Workload patterns:")
    plot_workload_patterns(n_hours=168, show=True)

    # Demo với episode giả lập
    print("\n3. Episode analysis demo:")

    env = SpotInstanceEnv()
    obs, _ = env.reset(seed=42)

    # Chạy một episode với random policy
    done = False
    while not done:
        action = env.action_space.sample()
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

    history = env.get_history()
    create_dashboard(history, policy_name="Random Policy", show=True)

    env.close()


In [ ]:
# Run all cells above first
print('Lab initialized.')
